# Structured Semantics to Speech

**Assumes notebook 01 (`image_to_text_finetuning`) has already produced a fine-tuned BLIP checkpoint.**

This notebook:
1. Loads the fine-tuned BLIP captioner.
2. Generates **structured semantics** (title, caption, year, period, `suggested_tone`) for each painting.
3. Synthesises a **spoken description** with `edge-tts`, picking the voice and prosody style
   automatically from `semantics["suggested_tone"]` (`"calm"` or `"engaging"`).

Output: one `.mp3` per painting saved to Google Drive.

## 1. Check GPU

Enable GPU before running: **Runtime → Change runtime type → T4 GPU** (or better).

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the Repo

Replace `USERNAME` with your GitHub username and `BRANCH_NAME` with your working branch (e.g. `elpida`).

In [ ]:
# First-time clone — replace USERNAME and BRANCH_NAME before running.
!git clone -b BRANCH_NAME https://github.com/USERNAME/mscai-multimodal-project.git /content/mscai-multimodal-project
%cd /content/mscai-multimodal-project

# --- Already cloned? Use this instead (comment out the clone above): ---
# %cd /content/mscai-multimodal-project
# !git pull

## 4. Install Dependencies

Install in two stages to avoid dependency conflicts: LAVIS pins its own core libraries
(`transformers <4.27`, `fairscale 0.4.x`, old `huggingface_hub`) and must be installed
first; adding all packages at once causes `ResolutionImpossible` errors.

**After running this cell, a single "Restart runtime" is required
(Runtime → Restart session), then continue from the next cell.
Do NOT re-run this install cell after restarting.**

In [ ]:
# Stage 1: install LAVIS — this pulls and pins its required core deps
# (transformers <4.27, fairscale 0.4.x, old huggingface_hub, timm, etc.).
!pip install -e ./LAVIS --quiet

# Stage 2: lightweight extras added AFTER LAVIS pins are in place.
# These packages are tolerant of the versions LAVIS already installed.
# Fallback note: if 'datasets' forces an incompatible huggingface_hub, install it
# separately with: pip install datasets==2.14.6 --no-deps
!pip install pycocoevalcap mlflow scikit-learn matplotlib datasets --quiet

# TTS stack — not part of LAVIS; safe to install last.
!pip install edge-tts nest_asyncio --quiet

## 5. Configure Paths

Adjust `DATA_DIR` to match your Drive layout and `CHECKPOINT_PATH` to the `.pth` saved by notebook 01.

In [ ]:
# Root data folder on Drive — adjust to your path.
DATA_DIR = "/content/drive/MyDrive/Github/mscai-multimodal-project/data"

# Fine-tuned checkpoint from train_caption.py — adjust epoch number as needed.
CHECKPOINT_PATH = f"{DATA_DIR}/../checkpoints/blip_artpedia_epoch5.pth"

print(f"DATA_DIR        : {DATA_DIR}")
print(f"CHECKPOINT_PATH : {CHECKPOINT_PATH}")

## 6. Load Fine-tuned Model and Dataset

Builds a `Captioner` with the fine-tuned checkpoint (prints missing/unexpected key counts
so you can sanity-check the load) and the Artpedia test split.

In [ ]:
import sys
sys.path.insert(0, 'src')

from captioner import Captioner
from artpedia_dataset import ArtpediaDataset

# Load fine-tuned BLIP captioner.
captioner = Captioner(checkpoint_path=CHECKPOINT_PATH)

# Artpedia test split — provides records and cached images.
dataset = ArtpediaDataset('dataset/artpedia/artpedia.json', split='test')
print(f"Test split: {len(dataset)} records")

## 7. Generate Structured Semantics

For each of the first `N` paintings, the pipeline:
1. Loads the cached image from Drive.
2. Generates a caption with the fine-tuned `Captioner`.
3. Calls `build_semantics(record, caption)` to produce a dict with keys
   `title`, `caption`, `year`, `period`, `suggested_tone`, `source`.

The `suggested_tone` (`"calm"` or `"engaging"`) is automatically derived from the
painting's period and drives voice selection in the next section.

In [ ]:
import json
from structured_semantics import build_semantics

N = 5  # number of paintings to process; increase as needed

all_semantics = []

for i in range(N):
    record = dataset[i]

    # Load the locally cached image (no network call).
    image = dataset.load_image(i)

    # Generate one caption with the fine-tuned model.
    caption = captioner.caption_image(image, num_captions=1)[0]

    # Build structured semantics; suggested_tone is derived from the period.
    sem = build_semantics(record, caption)
    all_semantics.append(sem)

    print(f"--- Painting {i} ---")
    print(json.dumps(sem, indent=2, ensure_ascii=False))
    print()

## 8. TTS Setup

Two prosody styles matching the project's TTS spec:
- **calm** — `en-GB-SoniaNeural`, slower and lower-pitched (classical/older works).
- **engaging** — `en-US-JennyNeural`, warmer and slightly elevated pitch (dynamic works).

Output mp3s are saved to Drive so they persist across sessions.

In [ ]:
import edge_tts
import nest_asyncio
import asyncio
import os

nest_asyncio.apply()  # lets `await` work inside Colab's existing event loop

# Output folder on Drive.
TTS_OUTPUT_DIR = f"{DATA_DIR}/tts_outputs"
os.makedirs(TTS_OUTPUT_DIR, exist_ok=True)
print(f"TTS output dir: {TTS_OUTPUT_DIR}")

# Tone definitions — strings must match suggested_tone values: "calm" / "engaging".
STYLES = {
    "calm": {
        "voice":  "en-GB-SoniaNeural",
        "rate":   "-5%",
        "pitch":  "-2Hz",
        "volume": "+0%",
    },
    "engaging": {
        "voice":  "en-US-JennyNeural",
        "rate":   "-2%",
        "pitch":  "+4Hz",
        "volume": "+3%",
    },
}

## 9. Synthesise Speech

`build_spoken_text` composes the spoken text from the semantics dict:
it prepends an optional intro (`<Title>. <Period>, circa <year>.`) built only from
fields that are present and meaningful, then appends the generated caption.
If no metadata intro can be built, only the caption is spoken.

Tone selection from `suggested_tone` and all voice/prosody settings are unchanged.

In [ ]:
def build_spoken_text(sem: dict) -> str:
    """Compose spoken text: optional metadata intro + caption."""
    title  = sem.get("title", "")
    period = sem.get("period", "")
    year   = sem.get("year")

    intro_parts = []
    if title and title != "Untitled":
        intro_parts.append(title + ".")

    meta = []
    if period and period != "Unknown":
        meta.append(period)
    if year is not None:
        meta.append(f"circa {year}")
    if meta:
        intro_parts.append(", ".join(meta) + ".")

    intro   = " ".join(intro_parts)
    caption = sem.get("caption", "")
    return f"{intro} {caption}".strip() if intro else caption


async def synthesise_all(semantics_list):
    for idx, sem in enumerate(semantics_list):
        # Tone is set by the semantics pipeline; fall back to calm if unrecognised.
        tone = sem.get("suggested_tone", "calm")
        if tone not in STYLES:
            print(f"  [WARN] Unknown tone '{tone}' for index {idx} — using 'calm'.")
            tone = "calm"

        style    = STYLES[tone]
        out_path = os.path.join(TTS_OUTPUT_DIR, f"{idx}_{tone}.mp3")

        communicate = edge_tts.Communicate(
            text=build_spoken_text(sem),
            voice=style["voice"],
            rate=style["rate"],
            pitch=style["pitch"],
            volume=style["volume"],
        )
        await communicate.save(out_path)
        print(f"  [{tone}] saved: {out_path}")


await synthesise_all(all_semantics)

## 10. Play a Sample Inline (Optional)

Change `PLAY_INDEX` to listen to any of the generated files directly in the notebook.

In [ ]:
from IPython.display import Audio

PLAY_INDEX = 0
play_tone  = all_semantics[PLAY_INDEX].get("suggested_tone", "calm")
mp3_path   = os.path.join(TTS_OUTPUT_DIR, f"{PLAY_INDEX}_{play_tone}.mp3")

print(f"Playing: {mp3_path}")
Audio(mp3_path)